# 使用 LlamaFactory CLI 进行 LoRA 微调、推理与权重合并

本 Notebook 以**命令行示意**为主，完整演示从数据准备到模型合并的全流程。

---
**硬件需求**：显存 ≥ 16GB（7B 模型 + LoRA + bf16），推荐 A100 / 3090 / 4090  
**框架版本**：LLaMA Factory ≥ 0.9.0

## 一、环境安装

### 1.1 安装 LLaMA Factory

克隆官方仓库并以可编辑模式安装，`[torch,metrics]` 会同时安装训练和评估所需的依赖：

```bash
git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
cd LLaMA-Factory
pip install -e ".[torch,metrics]"
```

### 1.2 验证安装

```bash
llamafactory-cli version
nvidia-smi
```

## 二、数据集准备

### 2.1 数据格式

LLaMA Factory 支持两种主流数据格式，按数据来源和对话轮次选择：

---

#### 格式一：Alpaca 格式（单轮，推荐用于指令微调）

适合**单轮问答**场景，每条样本由 instruction + input + output 三部分构成。  
保存到 `data/my_alpaca.json`：

```json
[
  {
    "instruction": "请将以下文本翻译成英文",
    "input": "人工智能正在改变世界。",
    "output": "Artificial intelligence is changing the world."
  },
  {
    "instruction": "写一首关于春天的五言绝句",
    "input": "",
    "output": "春风吹绿岸，燕子衔泥忙。花开千树艳，鸟鸣一声长。"
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `instruction` | 必填 | 任务指令，最终拼接为 prompt |
| `input` | 可为空字符串 | 用户的具体输入内容，为空时只用 instruction |
| `output` | 必填 | 期望的模型回答 |
| `system` | 可选 | 为该条样本单独指定 system prompt |

---

#### 格式二：ShareGPT 格式（多轮，推荐用于对话微调）

适合**多轮对话**场景，每条样本包含一段完整的对话历史。  
保存到 `data/my_sharegpt.json`：

```json
[
  {
    "system": "你是一个有帮助的中文AI助手。",
    "conversations": [
      {
        "from": "human",
        "value": "你好，请介绍一下大语言模型。"
      },
      {
        "from": "gpt",
        "value": "大语言模型（LLM）是基于 Transformer 架构、在海量文本上预训练的神经网络。"
      },
      {
        "from": "human",
        "value": "它和普通的神经网络有什么区别？"
      },
      {
        "from": "gpt",
        "value": "主要区别在于规模和预训练范式：LLM 参数量通常在 10亿以上，且通过自监督方式在互联网级语料上训练。"
      }
    ]
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `system` | 可选 | 全局 system prompt，适用于该条对话的所有轮次 |
| `conversations` | 必填 | 对话轮次列表，按时间顺序排列 |
| `conversations[i].from` | 必填 | 发言角色，`human` 表示用户，`gpt` 表示模型 |
| `conversations[i].value` | 必填 | 该轮次的具体文本内容 |

> **如何选择格式：**  
> - 数据来自人工标注的单轮问答 → Alpaca 格式  
> - 数据来自 ChatGPT / Claude 的多轮对话导出 → ShareGPT 格式  
> - 两种格式可以在 `dataset` 字段中混用，框架自动识别

### 2.2 注册数据集

编辑 `data/dataset_info.json`，两种格式的注册方式略有不同：

**Alpaca 格式注册**（通过 `columns` 映射字段名）：

```json
{
  "my_alpaca": {
    "file_name": "my_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "system":   "system"
    }
  }
}
```

**ShareGPT 格式注册**（需额外声明 `formatting` 和 `tags` 定义角色标识）：

```json
{
  "my_sharegpt": {
    "file_name": "my_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "system":   "system"
    },
    "tags": {
      "role_tag":      "from",
      "content_tag":   "value",
      "user_tag":      "human",
      "assistant_tag": "gpt"
    }
  }
}
```

| 字段 | 说明 |
|---|---|
| `formatting` | Alpaca 格式省略此字段；ShareGPT 格式必须填 `"sharegpt"` |
| `columns.messages` | JSON 中对话列表对应的字段名（本例为 `conversations`） |
| `tags.role_tag` | 每条消息中标识角色的键名（本例为 `from`） |
| `tags.user_tag` | 用户角色的值（本例为 `human`） |
| `tags.assistant_tag` | 模型角色的值（本例为 `gpt`） |

---

> **`file_name` 路径说明**
>
> | 写法 | 起点 | 示例 | 适用场景 |
> |---|---|---|---|
> | 相对路径 | `dataset_info.json` 所在目录（默认 `data/`） | `"my_alpaca.json"` → `data/my_alpaca.json` | 数据集在 `data/` 目录下（推荐） |
> | `..` 相对路径 | 同上，通过 `..` 向上跳出 `data/` | `"../raw/train.json"` → 项目根下的 `raw/train.json` | 可用，但路径深时难以维护 |
> | **绝对路径** | **文件系统根目录，与 `data/` 完全无关** | `"/home/user/datasets/train.json"` | 数据集在 `data/` 之外时推荐使用 |
>
> 绝对路径**不需要经过 `data/` 目录**，直接从磁盘根（Linux: `/`，Windows: `C:\`）出发定位文件，框架检测到绝对路径后会跳过拼接直接加载。

---

注册后在训练配置的 `dataset` 字段中填入**键名**（即 `dataset_info.json` 中的 JSON key，不是文件名），**多个数据集可以用逗号拼接混用**：

```yaml
# 填写的是 dataset_info.json 中的键名（my_alpaca / my_sharegpt），而非文件名（my_alpaca.json）
dataset: my_alpaca,my_sharegpt   # 框架根据键名查找注册信息，自动合并并打乱顺序
```

## 三、LoRA 微调训练

### 3.1 修改训练配置文件

训练配置文件位于 `configs/lora_LlamaFactory_train.yaml`，启动训练前修改以下**必填字段**：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型本地路径或 HuggingFace Hub ID |
| `template` | 对话模板，需与模型一致（如 `qwen` / `chatglm3` / `llama3`） |
| `dataset` | 在 `dataset_info.json` 中注册的数据集名称 |
| `output_dir` | 训练结果（LoRA 权重 + 检查点）的保存路径 |

其余参数（LoRA rank/alpha、学习率、早停等）均已配有详细注释，按需调整。

### 3.2 启动训练

**单卡训练**

```bash
llamafactory-cli train configs/lora_LlamaFactory_train.yaml
```

**多卡分布式训练（DeepSpeed）**

```bash
# --num_gpus 指定 GPU 卡数；deepspeed 字段在 YAML 中指定 Zero 策略配置文件
deepspeed --num_gpus=4 \
    $(which llamafactory-cli) train configs/lora_LlamaFactory_train.yaml
```

在训练配置文件中取消注释 `deepspeed` 字段以启用对应的 Zero 优化策略：

```yaml
# Zero-2：优化器状态 + 梯度跨卡分片，模型参数每卡完整保留（适合大多数场景）
deepspeed: examples/deepspeed/ds_z2_config.json

# Zero-3：在 Zero-2 基础上进一步将模型参数也跨卡分片（适合超大模型显存不足时）
# deepspeed: examples/deepspeed/ds_z3_config.json
```

> `examples/deepspeed/` 目录为 LLaMA Factory 仓库内置的配置文件，克隆仓库后即可直接使用。

**训练完成后，`output_dir` 目录结构如下：**

```
output/lora_sft/
├── checkpoint-500/                  # 中间检查点
├── adapter_model.safetensors        # 最终 LoRA 权重（仅几十 MB）
├── adapter_config.json              # LoRA 结构配置
├── trainer_state.json               # 训练历史与最优检查点信息
└── training_loss.png                # 训练 / 验证损失曲线图
```

> **判断训练质量：** train_loss 和 eval_loss 同步下降 → 正常；eval_loss 反弹而 train_loss 继续下降 → 过拟合，减少 epoch 或增大 lora_dropout。

## 四、推理（Chat 模式）

### 4.1 准备推理配置文件

新建 `configs/lora_LlamaFactory_infer.yaml`，填入以下关键字段：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # 训练输出的 LoRA 权重目录
finetuning_type: lora                          # 告知框架加载 LoRA 适配器
template: qwen                                 # 须与训练时保持一致

# 生成参数
max_new_tokens: 512
temperature: 0.7
top_p: 0.9
```

框架会自动在内存中将 LoRA 适配器合并到基础模型再进行推理，**无需提前合并权重**。

### 4.2 启动交互式对话

```bash
llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
```

运行后在终端直接输入问题，框架返回模型回答，输入 `exit` 或按 `Ctrl+C` 退出。

**也可以启动 Web UI 进行可视化对话：**

```bash
llamafactory-cli webui
# 打开浏览器访问 http://localhost:7860
# 在 Chat 标签页填入模型路径和 LoRA 路径后即可对话
```

## 五、合并 LoRA 权重

### 5.1 合并原理

LoRA 的低秩矩阵 $B \cdot A$ 乘以缩放系数后直接加回原始权重，得到与原模型结构完全相同的完整模型：

$$W_{merged} = W_{base} + \frac{\alpha}{r} \cdot B \cdot A$$

| | 合并前（LoRA 动态加载） | 合并后（完整模型） |
|---|---|---|
| 磁盘占用 | 基础模型 + 几十 MB | 与基础模型等大 |
| 推理延迟 | 略高（运行时合并） | 与原始模型相同 |
| 适用场景 | 训练调试、快速切换多个 LoRA | 生产部署、进一步量化 |

### 5.2 准备合并配置文件

新建 `configs/lora_LlamaFactory_merge.yaml`：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # LoRA 权重目录
finetuning_type: lora
template: llama3

export_dir: ./output/merged_model              # 合并后完整模型的保存路径
export_size: 2                                 # 每个分片文件最大 2 GB
export_device: cuda                            # 合并运算设备（显存不足时改为 cpu）
export_legacy_format: false                    # false=保存为 safetensors（推荐）
```

### 5.3 执行合并

```bash
llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
```

合并完成后，`export_dir` 目录结构如下，可直接作为标准 HuggingFace 模型使用：

```
output/merged_model/
├── model-00001-of-00003.safetensors   # 合并后完整权重（分片）
├── model-00002-of-00003.safetensors
├── model-00003-of-00003.safetensors
├── model.safetensors.index.json
├── config.json
├── tokenizer.json
└── tokenizer_config.json
```

**验证合并后模型可正常推理：**

```bash
# 将推理配置中的 model_name_or_path 改为合并后目录，去掉 adapter_name_or_path
llamafactory-cli chat configs/lora_LlamaFactory_merged_infer.yaml
```

## 附：完整工作流总结

```
原始数据（JSON）
   │
   ▼ 二、注册到 data/dataset_info.json
   │
   ▼ 三、修改 configs/lora_LlamaFactory_train.yaml
   │
   ▼ llamafactory-cli train configs/lora_LlamaFactory_train.yaml
   │
   └─→ output/lora_sft/
           ├── adapter_model.safetensors   ← LoRA 权重
           └── training_loss.png
   │
   ├─▶ 四、推理（LoRA 动态加载）
   │    llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
   │
   ▼ 五、llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
   │
   └─→ output/merged_model/   ← 标准 HuggingFace 完整模型
```

**下一步建议**
- 使用 `vllm` 或 `llama.cpp` 对合并后的模型进行量化部署
- 使用 `llamafactory-cli eval` 在 MMLU / C-Eval 等基准上评测效果
- 尝试 DPO / KTO 阶段对齐微调，提升模型的指令跟随能力